<a href="https://colab.research.google.com/github/InoriNatsume/anima_colab_test/blob/main/anima_lora_train_kohya_gui_only_colab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎨 Anima LoRA Training (kohya_ss_anima + sd-scripts / Colab T4, GUI-only)

> **Anima** — 2B parameter anime-focused text-to-image model (Cosmos-Predict2 기반, CircleStone Labs)
>
> [kohya_ss](https://github.com/gabengaGamer/kohya_ss_anima) + [sd-scripts](https://github.com/kohya-ss/sd-scripts) 기반 Anima LoRA 학습 노트북
>
> **환경:** Colab T4 (16GB VRAM) / bf16 / Prodigy / LoRA
>
> 주요 하이퍼파라미터는 셀 4-1, 4-2의 `@param` 폼에서 조절 가능합니다.

### 고정 설정 (변경 비권장)
| 항목 | 값 | 이유 |
|---|---|---|
| dtype | `bfloat16` | T4에서 안정적인 학습을 위한 기본값 |
| gradient_checkpointing | `true` | VRAM 사용량 절감 |
| cache_latents | `true` | 반복 학습 시 속도 개선 |


## 1. 환경 설정 및 의존성 설치

In [ ]:
#@title 1-1. GPU 확인 및 CUDA 메모리 설정
import torch, subprocess

# GPU 확인
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
print(f"🖥️  GPU: {gpu_name}")
print(f"💾 VRAM: {gpu_mem:.1f} GB")
print(f"🔧 CUDA: {torch.version.cuda}")
print(f"🐍 PyTorch: {torch.__version__}")

# bf16 지원 여부 확인
bf16_support = torch.cuda.is_bf16_supported()
print(f"📋 BF16 지원: {bf16_support}")
if bf16_support:
    print("   → BF16 모드로 학습합니다")
else:
    print("   ⚠️ BF16 미지원 — TOML에서 dtype을 float16으로 수정 필요")

# CUDA 메모리 단편화 방지
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# ========================================
# 🤗 HuggingFace Hub 인증 (Colab 시크릿)
# ========================================
# Colab 시크릿에 'HF_upload' 이름으로 HuggingFace 토큰(Write 권한)을 등록하세요.
# 설정 방법: Colab 왼쪽 🔑 아이콘 → 새 시크릿 추가 → 이름: HF_upload, 값: hf_xxxxx
HF_TOKEN = None
HF_USERNAME = None

# Step 1: Colab 시크릿에서 토큰 읽기
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_upload')
except userdata.SecretNotFoundError:
    print("\n❌ HF_upload 시크릿이 등록되지 않았습니다!")
    print("   📌 등록 방법:")
    print("   1. Colab 왼쪽 사이드바에서 🔑 (시크릿) 아이콘 클릭")
    print("   2. '새 시크릿 추가' 클릭")
    print("   3. 이름: HF_upload")
    print("   4. 값: hf_xxxxxxxxxxxx (HuggingFace 토큰)")
    print("   5. '노트북 액세스' 토글 ON")
    print("   6. 이 셀을 다시 실행")
    print("\n   🔗 토큰 발급: https://huggingface.co/settings/tokens")
except userdata.NotebookAccessError:
    print("\n❌ HF_upload 시크릿의 '노트북 액세스'가 꺼져 있습니다!")
    print("   📌 왼쪽 🔑 아이콘 → HF_upload → '노트북 액세스' 토글 ON → 이 셀 다시 실행")
except Exception as e:
    print(f"\n⚠️ 시크릿 읽기 실패: {e}")

# Step 2: 토큰 형식 검증
if HF_TOKEN:
    HF_TOKEN = HF_TOKEN.strip()
    if not HF_TOKEN.startswith('hf_'):
        print(f"\n❌ 토큰 형식이 잘못되었습니다! (현재: '{HF_TOKEN[:10]}...')")
        print("   HuggingFace 토큰은 'hf_'로 시작해야 합니다.")
        print("   🔗 올바른 토큰 발급: https://huggingface.co/settings/tokens")
        HF_TOKEN = None

# Step 3: 토큰으로 인증 시도
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi, login
        login(token=HF_TOKEN, add_to_git_credential=True)
        api = HfApi()
        user_info = api.whoami(token=HF_TOKEN)
        HF_USERNAME = user_info["name"]

        # Write 권한 확인
        auth = user_info.get("auth", {})
        access_token = auth.get("accessToken", {})
        role = access_token.get("role", "unknown")
        if role == "read":
            print(f"\n⚠️ 토큰에 Write 권한이 없습니다! (현재: read-only)")
            print("   📌 https://huggingface.co/settings/tokens 에서")
            print("   'Write' 권한의 토큰을 새로 발급하세요.")
            HF_TOKEN = None
            HF_USERNAME = None
        else:
            print(f"\n🤗 HuggingFace 인증 성공!")
            print(f"   👤 사용자: {HF_USERNAME}")
            print(f"   🔑 토큰: hf_...{HF_TOKEN[-4:]}")
    except Exception as e:
        error_msg = str(e)
        if "401" in error_msg or "Unauthorized" in error_msg:
            print(f"\n❌ 토큰이 유효하지 않습니다! (만료되었거나 잘못된 값)")
            print("   📌 https://huggingface.co/settings/tokens 에서 토큰을 확인하세요.")
        else:
            print(f"\n⚠️ HF 인증 실패: {e}")
        HF_TOKEN = None
        HF_USERNAME = None

if not HF_TOKEN:
    print("\n   → HF 업로드 없이 학습만 진행합니다. (셀 7은 건너뛰세요)")

🖥️  GPU: Tesla T4
💾 VRAM: 14.6 GB
🔧 CUDA: 12.8
🐍 PyTorch: 2.10.0+cu128
📋 BF16 지원: True
   → BF16 모드로 학습합니다

🤗 HuggingFace 인증 성공!
   👤 사용자: Kamome33
   🔑 토큰: hf_...YKHM


In [ ]:
#@title 1-2. kohya_ss_anima 클론 및 의존성 설치
import os
import subprocess

# Anima enabled kohya fork URL
KOHYA_REPO_URL = "https://github.com/gabengaGamer/kohya_ss_anima.git"  #@param {type:"string"}
KOHYA_DIR = "/content/kohya_ss_anima"  #@param {type:"string"}

if not os.path.exists(KOHYA_DIR):
    subprocess.check_call(["git", "clone", KOHYA_REPO_URL, KOHYA_DIR])
else:
    print(f"already exists: {KOHYA_DIR}")

os.chdir(KOHYA_DIR)

anima_script = os.path.join(KOHYA_DIR, "sd-scripts", "anima_train_network.py")
if not os.path.exists(anima_script):
    raise RuntimeError(
        "This repository does not include sd-scripts/anima_train_network.py. "
        "Please use an Anima-enabled kohya_ss_anima fork."
    )

# 1) Install upstream dependencies as-is
subprocess.check_call(["pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call(["pip", "install", "-q", "bitsandbytes", "tensorboard"])

# 2) Minimal compatibility fixes for Colab runtime
#    - avoid NumPy 2.x ABI breakage
#    - keep huggingface-hub compatible with transformers 4.54.x
compat_specs = ["numpy<2", "huggingface-hub>=0.34,<1.0"]
subprocess.check_call(["pip", "install", "-q", "--upgrade"] + compat_specs)

# Environment snapshot for reproducibility
with open('/content/pip_freeze.txt', 'w', encoding='utf-8') as f:
    subprocess.check_call(["python", "-m", "pip", "freeze"], stdout=f)

print("install done")
print("saved: /content/pip_freeze.txt")


install done
saved: /content/pip_freeze.txt


## 2. 모델 다운로드 (HuggingFace → Colab)

Anima 모델 3개 파일을 HF에서 직접 Colab 로컬 스토리지로 다운로드합니다.
- **anima-preview.safetensors** — Transformer/DiT (~4.2GB)
- **qwen_3_06b_base.safetensors** — 텍스트 인코더 Qwen3-0.6B (~1.2GB)
- **qwen_image_vae.safetensors** — VAE (~0.2GB)

In [ ]:
#@title 2-1. Anima 모델 파일 다운로드
from huggingface_hub import hf_hub_download
import os

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

REPO_ID = "circlestone-labs/Anima"

# Transformer (DiT) — anima-preview.safetensors
transformer_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="split_files/diffusion_models/anima-preview.safetensors",
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,
)
# 실제 저장 경로 정리
TRANSFORMER_PATH = os.path.join(MODEL_DIR, "split_files/diffusion_models/anima-preview.safetensors")
print(f"✅ Transformer: {TRANSFORMER_PATH}")
print(f"   크기: {os.path.getsize(TRANSFORMER_PATH)/1024**3:.2f} GB")

# Text Encoder — qwen_3_06b_base.safetensors
text_enc_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="split_files/text_encoders/qwen_3_06b_base.safetensors",
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,
)
LLM_PATH = os.path.join(MODEL_DIR, "split_files/text_encoders/qwen_3_06b_base.safetensors")
print(f"✅ Text Encoder: {LLM_PATH}")
print(f"   크기: {os.path.getsize(LLM_PATH)/1024**3:.2f} GB")

# VAE — qwen_image_vae.safetensors
vae_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="split_files/vae/qwen_image_vae.safetensors",
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,
)
VAE_PATH = os.path.join(MODEL_DIR, "split_files/vae/qwen_image_vae.safetensors")
print(f"✅ VAE: {VAE_PATH}")
print(f"   크기: {os.path.getsize(VAE_PATH)/1024**3:.2f} GB")

print(f"\n📦 모델 총 크기: {sum(os.path.getsize(p) for p in [TRANSFORMER_PATH, LLM_PATH, VAE_PATH])/1024**3:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  `use_auth_token` will definitely not be supported.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


split_files/diffusion_models/anima-previ(…):   0%|          | 0.00/4.18G [00:00<?, ?B/s]

✅ Transformer: /content/models/split_files/diffusion_models/anima-preview.safetensors
   크기: 3.89 GB


split_files/text_encoders/qwen_3_06b_bas(…):   0%|          | 0.00/1.19G [00:00<?, ?B/s]

✅ Text Encoder: /content/models/split_files/text_encoders/qwen_3_06b_base.safetensors
   크기: 1.11 GB


split_files/vae/qwen_image_vae.safetenso(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

✅ VAE: /content/models/split_files/vae/qwen_image_vae.safetensors
   크기: 0.24 GB

📦 모델 총 크기: 5.24 GB


## 3. Google Drive 마운트 및 데이터셋 준비

데이터셋 구조: 이미지 파일 + 동일 이름의 `.txt` 캡션 파일  
캡션 형식: Danbooru 태그 (예: `shirakawa yuina, @artist, 1girl, solo, ...`)

**데이터셋 경로 입력 방식:**
- Google Drive: `/content/drive/MyDrive/yuina_anima` (절대경로)
- HuggingFace: `username/repo_name` (`/`로 시작하지 않으면 HF 리포로 인식)

In [ ]:
#@title 3-1. 데이터셋 준비 (Google Drive 또는 HuggingFace)
import shutil, glob, os, zipfile

# Inputs
DATASET_SOURCE = ""  #@param {type:"string"}
GDRIVE_OUTPUT_DIR = ""  #@param {type:"string"}
HF_REPO_NAME = ""  #@param {type:"string"}

LOCAL_DATASET_DIR = "/content/dataset"
if os.path.exists(LOCAL_DATASET_DIR):
    shutil.rmtree(LOCAL_DATASET_DIR)
os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)

source_input = DATASET_SOURCE.strip()
hf_repo_name_input = HF_REPO_NAME.strip()

# Build source list:
# - DATASET_SOURCE if provided
# - HF_REPO_NAME if it looks like owner/repo
source_candidates = []
if source_input:
    source_candidates.append(("dataset_source", source_input))

hf_repo_name_used_as_source = False
if hf_repo_name_input and (not hf_repo_name_input.startswith('/')) and ('/' in hf_repo_name_input):
    if hf_repo_name_input != source_input:
        source_candidates.append(("hf_repo_name", hf_repo_name_input))
    hf_repo_name_used_as_source = True

if not source_candidates:
    raise RuntimeError(
        "No dataset source provided. Fill DATASET_SOURCE, or put owner/repo in HF_REPO_NAME."
    )


def _normalize_hf_repo_id(src: str) -> str:
    s = src.strip()
    if s.startswith("https://huggingface.co/"):
        s = s.replace("https://huggingface.co/", "").strip("/")
    if s.startswith("hf://"):
        s = s.replace("hf://", "").strip("/")
    if s.startswith("datasets/"):
        s = s[len("datasets/"):]
    return s


def _merge_dir(src_root: str, dst_root: str, skip_hidden: bool):
    for name in os.listdir(src_root):
        if skip_hidden and name.startswith('.'):
            continue
        src_path = os.path.join(src_root, name)
        dst_path = os.path.join(dst_root, name)
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
        else:
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(src_path, dst_path)


def _load_from_hf(repo_id: str):
    repo_id = _normalize_hf_repo_id(repo_id)
    if '/' not in repo_id:
        raise RuntimeError(f"Invalid HF repo id: {repo_id}")

    print(f"[HF] source: {repo_id}")

    # Compatibility shim for occasional huggingface_hub/hf_transfer mismatch in Colab runtime
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
    try:
        import huggingface_hub.constants as _hh_constants
        if not hasattr(_hh_constants, "HF_HUB_ENABLE_HF_TRANSFER"):
            _hh_constants.HF_HUB_ENABLE_HF_TRANSFER = False
    except Exception:
        pass

    from huggingface_hub import HfApi, snapshot_download

    api = HfApi()
    dataset_err = None
    model_err = None

    # 1) First, explicitly check dataset repo.
    try:
        api.repo_info(repo_id=repo_id, repo_type="dataset", token=HF_TOKEN if HF_TOKEN else None)
        target_repo_type = "dataset"
    except Exception as e:
        dataset_err = e
        # 2) Fallback: check model repo only if dataset lookup failed.
        try:
            api.repo_info(repo_id=repo_id, repo_type="model", token=HF_TOKEN if HF_TOKEN else None)
            target_repo_type = "model"
        except Exception as e2:
            model_err = e2
            raise RuntimeError(
                "HF repo lookup failed. "
                f"repo_id={repo_id} | dataset_err={dataset_err} | model_err={model_err}"
            )

    print(f"   resolved repo_type: {target_repo_type}")

    tmp_dir = f"/content/_hf_dataset_{repo_id.replace('/', '_')}"
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)

    try:
        snapshot_download(
            repo_id=repo_id,
            repo_type=target_repo_type,
            local_dir=tmp_dir,
            token=HF_TOKEN if HF_TOKEN else None,
        )
    except Exception as e:
        raise RuntimeError(
            "HF snapshot_download failed. "
            f"repo_id={repo_id} repo_type={target_repo_type} err={e}"
        )

    zip_files = glob.glob(os.path.join(tmp_dir, "**", "*.zip"), recursive=True)
    if zip_files:
        for z in zip_files:
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(LOCAL_DATASET_DIR)
        print(f"   extracted zip files: {len(zip_files)}")
    else:
        _merge_dir(tmp_dir, LOCAL_DATASET_DIR, skip_hidden=True)
        print("   copied repo contents")

    shutil.rmtree(tmp_dir, ignore_errors=True)
    return repo_id.split('/')[-1], f"hf:{repo_id}"


def _load_from_gd(path: str, mounted: bool):
    if not mounted:
        from google.colab import drive
        drive.mount('/content/drive')
        mounted = True

    if not os.path.exists(path):
        raise RuntimeError(f"Google Drive path not found: {path}")

    print(f"[GD] source: {path}")

    if os.path.isdir(path):
        _merge_dir(path, LOCAL_DATASET_DIR, skip_hidden=False)
        print("   copied folder contents")
    elif path.lower().endswith('.zip'):
        with zipfile.ZipFile(path, 'r') as zf:
            zf.extractall(LOCAL_DATASET_DIR)
        print("   extracted zip")
    else:
        raise RuntimeError("GD source must be a directory or zip file.")

    name = os.path.basename(path.rstrip('/')) or "dataset"
    return name, mounted, f"gd:{path}"


dataset_names = []
source_logs = []
gd_mounted = False

for _, src_item in source_candidates:
    if src_item.startswith('/'):
        name, gd_mounted, src_log = _load_from_gd(src_item, gd_mounted)
    else:
        name, src_log = _load_from_hf(src_item)
    dataset_names.append(name)
    source_logs.append(src_log)

# Keep source info for later cells (README, etc.)
DATASET_SOURCE = ", ".join(source_logs)

# Decide output repo name
if hf_repo_name_used_as_source:
    HF_REPO_NAME = ""
else:
    HF_REPO_NAME = hf_repo_name_input

if HF_REPO_NAME and '/' in HF_REPO_NAME:
    # owner/repo is a source format, not output repo name
    HF_REPO_NAME = ""

if not HF_REPO_NAME:
    base_name = dataset_names[0] if dataset_names else "dataset"
    HF_REPO_NAME = base_name if len(dataset_names) == 1 else f"{base_name}_merged"

print(f"[INFO] upload target repo: {HF_USERNAME}/{HF_REPO_NAME}")

# Dataset checks
images = sorted(glob.glob(os.path.join(LOCAL_DATASET_DIR, "*.png")) +
                glob.glob(os.path.join(LOCAL_DATASET_DIR, "*.jpg")) +
                glob.glob(os.path.join(LOCAL_DATASET_DIR, "*.jpeg")) +
                glob.glob(os.path.join(LOCAL_DATASET_DIR, "*.webp")))
captions = sorted(glob.glob(os.path.join(LOCAL_DATASET_DIR, "*.txt")))

print("")
print(f"[CHECK] dataset dir: {LOCAL_DATASET_DIR}")
print(f"[CHECK] images: {len(images)}")
print(f"[CHECK] captions: {len(captions)}")

if len(images) == 0:
    print("[WARN] no images found in dataset dir")
    print(f"[WARN] folder entries: {os.listdir(LOCAL_DATASET_DIR)[:10]}")

img_stems = {os.path.splitext(os.path.basename(f))[0] for f in images}
cap_stems = {os.path.splitext(os.path.basename(f))[0] for f in captions}
missing_cap = sorted(img_stems - cap_stems)
missing_img = sorted(cap_stems - img_stems)
if missing_cap or missing_img:
    print("[WARN] image-caption mismatch detected")
    for s in missing_cap:
        print(f"  missing caption: {s}.*")
    for s in missing_img:
        print(f"  missing image: {s}.txt")
else:
    print(f"[OK] image-caption pairs matched: {len(img_stems)}")

print("")
print("[PREVIEW] first captions")
for txt_file in captions[:3]:
    name = os.path.basename(txt_file)
    with open(txt_file, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    suffix = "..." if len(content) > 120 else ""
    print(f"  {name}: {content[:120]}{suffix}")

GDRIVE_OUTPUT_DIR = GDRIVE_OUTPUT_DIR.strip()
if GDRIVE_OUTPUT_DIR:
    os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)
    print(f"[INFO] GD backup dir: {GDRIVE_OUTPUT_DIR}")
else:
    print("[INFO] GD backup disabled")


[HF] source: Kamome33/cynthia_only
   resolved repo_type: dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  `use_auth_token` will definitely not be supported.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

cyn_only.zip:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

   extracted zip files: 1
[INFO] upload target repo: Kamome33/cynthia_only

[CHECK] dataset dir: /content/dataset
[CHECK] images: 15
[CHECK] captions: 15
[OK] image-caption pairs matched: 15

[PREVIEW] first captions
  5995466.txt: cynthia (pokemon), @shirasulatte, 1girl, peeing, pee, solo, squatting, fellatio gesture, pubic hair, cow print, animal e...
  6453914.txt: cynthia (pokemon), @shirasulatte, stray pubic hair, gokkun, cum in container, 1girl, solo, cum in mouth, cum, tongue, cu...
  6880936.txt: cynthia (pokemon), @shirasulatte, :>=, 1boy, censored, 1girl, penis, hetero, fellatio, oral, bar censor, nipples, veins,...
[INFO] GD backup disabled


## 4. JSON 기반 CLI 학습 (sd-scripts)

GUI 없이도 외부 JSON을 불러 TOML 생성/검증/학습을 진행할 수 있습니다.

- 흐름: JSON 선택 -> TOML 생성 -> 경로 검증 -> (선택) TensorBoard 시작 -> (선택) 학습 실행
- 이 섹션은 GUI 섹션(5)과 별개로 사용할 수 있습니다.


In [ ]:
#@title 4-1. JSON -> TOML -> 검증 -> TensorBoard -> 학습 (통합)
import os
import glob
import json
import shlex
import subprocess
from datetime import datetime

KOHYA_DIR = "/content/kohya_ss_anima"
SD_SCRIPTS_DIR = os.path.join(KOHYA_DIR, "sd-scripts")
TRAIN_SCRIPT = os.path.join(SD_SCRIPTS_DIR, "anima_train_network.py")

CLI_CONFIG_JSON = ""  #@param {type:"string"}
CLI_OUTPUT_DIR_OVERRIDE = ""  #@param {type:"string"}
CLI_LOGGING_DIR_OVERRIDE = ""  #@param {type:"string"}
CLI_TRAIN_DIR_OVERRIDE = ""  #@param {type:"string"}
EXTRA_TOML_LINES = ""  #@param {type:"string"}
EXTRA_CLI_ARGS = ""  #@param {type:"string"}
FORCE_SAVE_STATE = True  #@param {type:"boolean"}
START_TENSORBOARD = True  #@param {type:"boolean"}
TB_PORT = 6010  #@param {type:"integer"}
RUN_CLI_TRAIN = False  #@param {type:"boolean"}


def _load_json(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return json.load(f)
    except Exception:
        return None


def _pick_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


def _bool(v, d=False):
    if isinstance(v, bool):
        return v
    if v is None:
        return d
    if isinstance(v, str):
        return v.strip().lower() in {"1", "true", "yes", "y", "on"}
    return bool(v)


def _to_list_optimizer_args(v):
    if v is None:
        return []
    if isinstance(v, list):
        return [str(x) for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    try:
        return [x for x in shlex.split(s) if x.strip()]
    except Exception:
        return [x for x in s.split() if x.strip()]


def _toml_val(v):
    if isinstance(v, bool):
        return "true" if v else "false"
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return repr(v)
    if isinstance(v, list):
        vals = ", ".join(_toml_val(x) for x in v)
        return f"[ {vals} ]"
    s = str(v).replace("\\", "\\\\").replace('"', '\\"')
    return f'"{s}"'


def _write_toml(path, data, extra_lines=""):
    lines = []
    for k, v in data.items():
        if v is None:
            continue
        lines.append(f"{k} = {_toml_val(v)}")
    if extra_lines.strip():
        lines.append("")
        lines.append("# user extra lines")
        lines.extend(extra_lines.split("\n"))
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")


def _count_images(path):
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".avif", ".jxl"}
    c = 0
    if not os.path.isdir(path):
        return 0
    for root, _, files in os.walk(path):
        for fn in files:
            if os.path.splitext(fn)[1].lower() in exts:
                c += 1
    return c


# 1) pick JSON
json_candidates = []
if CLI_CONFIG_JSON.strip():
    json_candidates.append(CLI_CONFIG_JSON.strip())
if "GUI_CONFIG_TO_LOAD" in globals() and str(GUI_CONFIG_TO_LOAD).strip():
    json_candidates.append(str(GUI_CONFIG_TO_LOAD).strip())
json_candidates.extend(sorted(glob.glob("/content/last_*.json"), key=os.path.getmtime, reverse=True))
json_candidates.extend(sorted(glob.glob("/content/outputs/last_*.json"), key=os.path.getmtime, reverse=True))
json_candidates.extend(sorted(glob.glob("/content/kohya_ss_anima/outputs/last_*.json"), key=os.path.getmtime, reverse=True))

seen = set()
uniq = []
for p in json_candidates:
    if p not in seen:
        seen.add(p)
        uniq.append(p)
json_candidates = uniq

picked_json = _pick_existing(json_candidates)
if not picked_json:
    raise RuntimeError("No config json found. Set CLI_CONFIG_JSON or prepare last_*.json.")

cfg = _load_json(picked_json)
if not cfg:
    raise RuntimeError(f"Failed to load json: {picked_json}")

# 2) resolve paths
model_path = cfg.get("pretrained_model_name_or_path", "")
qwen3_path = cfg.get("anima_qwen3", "") or cfg.get("qwen3", "")
vae_path = cfg.get("anima_vae", "") or cfg.get("vae", "")
train_data_dir = (CLI_TRAIN_DIR_OVERRIDE.strip() if CLI_TRAIN_DIR_OVERRIDE.strip() else cfg.get("train_data_dir", ""))
output_dir = (CLI_OUTPUT_DIR_OVERRIDE.strip() if CLI_OUTPUT_DIR_OVERRIDE.strip() else cfg.get("output_dir", "")) or "/content/kohya_ss_anima/outputs"
logging_dir = (CLI_LOGGING_DIR_OVERRIDE.strip() if CLI_LOGGING_DIR_OVERRIDE.strip() else cfg.get("logging_dir", "")) or "/content/kohya_ss_anima/logs"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(logging_dir, exist_ok=True)

# 3) validate
errors = []
for label, p in [
    ("pretrained_model_name_or_path", model_path),
    ("qwen3", qwen3_path),
    ("vae", vae_path),
    ("train_data_dir", train_data_dir),
]:
    if not p or not os.path.exists(p):
        errors.append(f"{label} missing: {p}")

img_count = _count_images(train_data_dir)
if img_count == 0:
    errors.append(f"no images in train_data_dir: {train_data_dir}")

if errors:
    raise RuntimeError("Validation failed\n- " + "\n- ".join(errors))

# 4) build TOML
optimizer_args = _to_list_optimizer_args(cfg.get("optimizer_args", ""))
additional_parameters = str(cfg.get("additional_parameters", "") or "").strip()
xformers_mode = str(cfg.get("xformers", "") or "").strip().lower()

train_toml = {
    "pretrained_model_name_or_path": model_path,
    "qwen3": qwen3_path,
    "qwen3_max_token_length": int(cfg.get("anima_qwen3_max_token_length", cfg.get("qwen3_max_token_length", 512)) or 512),
    "vae": vae_path,
    "vae_chunk_size": int(cfg.get("anima_vae_chunk_size", cfg.get("vae_chunk_size", 16)) or 16),
    "train_data_dir": train_data_dir,
    "output_dir": output_dir,
    "logging_dir": logging_dir,
    "output_name": cfg.get("output_name", "last"),
    "save_model_as": cfg.get("save_model_as", "safetensors"),
    "save_precision": cfg.get("save_precision", "bf16"),
    "mixed_precision": cfg.get("mixed_precision", "bf16"),
    "train_batch_size": int(cfg.get("train_batch_size", 1) or 1),
    "max_train_steps": int(cfg.get("max_train_steps", 0) or 0),
    "max_train_epochs": int(cfg.get("max_train_epochs", cfg.get("epoch", 0)) or 0),
    "save_every_n_epochs": int(cfg.get("save_every_n_epochs", 1) or 1),
    "save_every_n_steps": int(cfg.get("save_every_n_steps", 0) or 0),
    "caption_extension": cfg.get("caption_extension", ".txt"),
    "cache_latents": _bool(cfg.get("cache_latents", True), True),
    "cache_latents_to_disk": _bool(cfg.get("cache_latents_to_disk", False), False),
    "seed": int(cfg.get("seed", 0) or 0),
    "learning_rate": float(cfg.get("learning_rate", 1.0) or 1.0),
    "unet_lr": float(cfg.get("unet_lr", cfg.get("learning_rate", 1.0)) or 1.0),
    "text_encoder_lr": float(cfg.get("text_encoder_lr", 0) or 0),
    "t5xxl_lr": float(cfg.get("t5xxl_lr", 0) or 0),
    "optimizer_type": cfg.get("optimizer", "Prodigy"),
    "optimizer_args": optimizer_args,
    "lr_scheduler": cfg.get("lr_scheduler", "constant"),
    "lr_scheduler_num_cycles": int(cfg.get("lr_scheduler_num_cycles", 1) or 1),
    "lr_scheduler_power": float(cfg.get("lr_scheduler_power", 1) or 1),
    "lr_warmup_steps": int(cfg.get("lr_warmup_steps", 0) or 0),
    "max_grad_norm": float(cfg.get("max_grad_norm", 1) or 1),
    "gradient_checkpointing": _bool(cfg.get("gradient_checkpointing", True), True),
    "gradient_accumulation_steps": int(cfg.get("gradient_accumulation_steps", 1) or 1),
    "max_data_loader_n_workers": int(cfg.get("max_data_loader_n_workers", 0) or 0),
    "resolution": str(cfg.get("max_resolution", cfg.get("resolution", "512,512")) or "512,512"),
    "enable_bucket": _bool(cfg.get("enable_bucket", True), True),
    "bucket_no_upscale": _bool(cfg.get("bucket_no_upscale", True), True),
    "bucket_reso_steps": int(cfg.get("bucket_reso_steps", 64) or 64),
    "min_bucket_reso": int(cfg.get("min_bucket_reso", 256) or 256),
    "max_bucket_reso": int(cfg.get("max_bucket_reso", 2048) or 2048),
    "network_module": "networks.lora_anima",
    "network_dim": int(cfg.get("network_dim", 32) or 32),
    "network_alpha": int(cfg.get("network_alpha", 32) or 32),
    "network_dropout": float(cfg.get("network_dropout", 0) or 0),
    "network_train_unet_only": True,
    "shuffle_caption": _bool(cfg.get("shuffle_caption", True), True),
    "loss_type": cfg.get("loss_type", "l2"),
    "huber_schedule": cfg.get("huber_schedule", "snr"),
    "huber_c": float(cfg.get("huber_c", 0.1) or 0.1),
    "huber_scale": float(cfg.get("huber_scale", 1.0) or 1.0),
    "noise_offset_type": cfg.get("noise_offset_type", "Original"),
    "timestep_sampling": cfg.get("anima_timestep_sampling", cfg.get("timestep_sampling", "sigmoid")),
    "discrete_flow_shift": float(cfg.get("anima_discrete_flow_shift", cfg.get("discrete_flow_shift", 1.0)) or 1.0),
    "sigmoid_scale": float(cfg.get("anima_sigmoid_scale", cfg.get("sigmoid_scale", 1.0)) or 1.0),
    "sample_every_n_epochs": int(cfg.get("sample_every_n_epochs", 0) or 0),
    "sample_prompts": cfg.get("sample_prompts", ""),
    "sample_sampler": cfg.get("sample_sampler", "euler_a"),
    "wandb_run_name": cfg.get("wandb_run_name", ""),
    "save_state": _bool(cfg.get("save_state", False), False),
    "save_state_on_train_end": _bool(cfg.get("save_state_on_train_end", False), False),
    "split_attn": _bool(cfg.get("anima_split_attn", cfg.get("split_attn", False)), False),
    "sdpa": _bool(cfg.get("sdpa", xformers_mode == "sdpa"), xformers_mode == "sdpa"),
}

if FORCE_SAVE_STATE:
    train_toml["save_state"] = True
    train_toml["save_state_on_train_end"] = True

if train_toml.get("max_train_steps", 0) <= 0:
    train_toml.pop("max_train_steps", None)
if train_toml.get("max_train_epochs", 0) <= 0:
    train_toml.pop("max_train_epochs", None)

ts = datetime.now().strftime("%Y%m%d-%H%M%S")
toml_path = os.path.join(output_dir, f"config_cli_{ts}.toml")
_write_toml(toml_path, train_toml, EXTRA_TOML_LINES)

print("[SUMMARY]")
print("- picked json:", picked_json)
print("- toml path:", toml_path)
print("- output_dir:", output_dir)
print("- logging_dir:", logging_dir)
print("- train_data_dir:", train_data_dir)
print("- image count:", img_count)
print("- save_state:", train_toml.get("save_state"), "/ save_state_on_train_end:", train_toml.get("save_state_on_train_end"))
if additional_parameters:
    print("- additional_parameters:", additional_parameters)
if EXTRA_CLI_ARGS.strip():
    print("- extra_cli_args:", EXTRA_CLI_ARGS.strip())

print("\n[TOML preview]")
with open(toml_path, "r", encoding="utf-8") as f:
    lines = f.read().splitlines()
for ln in lines[:80]:
    print(ln)
if len(lines) > 80:
    print("...")

# 5) TensorBoard (optional)
if START_TENSORBOARD:
    subprocess.run("pkill -f 'tensorboard --logdir' || true", shell=True)
    tb_log = "/content/tensorboard_cli.log"
    tb_cmd = ["tensorboard", "--logdir", logging_dir, "--host", "0.0.0.0", "--port", str(TB_PORT)]
    with open(tb_log, "w", encoding="utf-8") as f:
        tb_proc = subprocess.Popen(tb_cmd, stdout=f, stderr=subprocess.STDOUT)
    print(f"\nTensorBoard started. pid={tb_proc.pid}, port={TB_PORT}")
    try:
        from google.colab.output import eval_js
        tb_url = eval_js(f"google.colab.kernel.proxyPort({TB_PORT})")
        print("TensorBoard URL:", tb_url)
    except Exception:
        print(f"Open TensorBoard via Colab proxy (port {TB_PORT})")

# 6) Train (optional)
cmd = [
    "accelerate", "launch",
    "--num_cpu_threads_per_process", str(int(cfg.get("num_cpu_threads_per_process", 2) or 2)),
    TRAIN_SCRIPT,
    "--config_file", toml_path,
]
if additional_parameters:
    cmd.extend(shlex.split(additional_parameters))
if EXTRA_CLI_ARGS.strip():
    cmd.extend(shlex.split(EXTRA_CLI_ARGS.strip()))

print("\n[train command]")
print(" ".join(shlex.quote(x) for x in cmd))

if RUN_CLI_TRAIN:
    print("\nTraining start...")
    res = subprocess.run(cmd, cwd=KOHYA_DIR)
    print("Training exit code:", res.returncode)
    if res.returncode != 0:
        raise RuntimeError(f"Training failed (return code={res.returncode})")
else:
    print("\nRUN_CLI_TRAIN=False, so training was not started.")


[1] GUI 입력 경로
- transformer: /content/models/split_files/diffusion_models/anima-preview.safetensors (exists=True)
- qwen3: /content/models/split_files/text_encoders/qwen_3_06b_base.safetensors (exists=True)
- vae: /content/models/split_files/vae/qwen_image_vae.safetensors (exists=True)
- dataset_dir: /content/dataset (exists=True)
- output_dir: /content/training_output (exists=True)
- logging_dir: /content/training_logs (exists=True)

[2] GUI 저장 TOML 탐색
- toml files: 1
  /content/kohya_ss_anima/config example.toml

[3] TensorBoard 로그 탐색
- tensorboard files: 0

[4] 이어 학습용 state/config
- state dirs: 0
- run configs: 0
- latest state: None
- latest config: None


## 5. GUI 실행

5-1 셀을 실행한 뒤 출력된 `GUI URL`로 접속하세요.


In [ ]:
from pathlib import Path
p = Path("/content/kohya_ss_anima/kohya_gui/class_tensorboard.py")
t = p.read_text(encoding="utf-8")

t = t.replace("from easygui import msgbox", "def msgbox(msg=None, **kwargs):\n    print(msg)")
t = t.replace(
    'if not os.path.exists(logging_dir) or not os.listdir(logging_dir):\n            self.log.error(\n                "Error: logging folder does not exist or does not contain logs."\n            )\n            msgbox(msg="Error: logging folder does not exist or does not contain logs.")\n            return self.get_button_states(started=False)',
    'if not os.path.exists(logging_dir):\n            self.log.error("Error: logging folder does not exist.")\n            return self.get_button_states(started=False)\n        # 폴더가 비어 있어도 TensorBoard는 먼저 띄울 수 있어야 함'
)

p.write_text(t, encoding="utf-8")
print("patched:", p)


patched: /content/kohya_ss_anima/kohya_gui/class_tensorboard.py


In [ ]:
#@title 5-1. Kohya GUI 실행 + Cloudflare Tunnel (권장)
import os
import re
import time
import signal
import socket
import subprocess

KOHYA_DIR = "/content/kohya_ss_anima"
GUI_PORT = 7860  # start port
PORT_SCAN_TRIES = 20  #@param {type:"integer"}
GUI_CONFIG_TO_LOAD = ""  #@param {type:"string"}
DISABLE_REQUIREMENTS_VERIFY = True  #@param {type:"boolean"}
RETRY_WITHOUT_CONFIG_IF_FAIL = True  #@param {type:"boolean"}
URL_WAIT_SECONDS = 180  #@param {type:"integer"}

GUI_LOG_PATH = "/content/kohya_gui.log"
GUI_PID_PATH = "/content/kohya_gui.pid"
CF_LOG_PATH = "/content/cloudflared.log"
CF_PID_PATH = "/content/cloudflared.pid"


def read_text(path: str) -> str:
    if not os.path.exists(path):
        return ""
    return open(path, "r", encoding="utf-8", errors="ignore").read()


def stop_process_by_pid_file(pid_path: str):
    if not os.path.exists(pid_path):
        return
    try:
        pid = int(open(pid_path, "r", encoding="utf-8").read().strip())
        os.kill(pid, signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass


def stop_prev_processes():
    stop_process_by_pid_file(GUI_PID_PATH)
    stop_process_by_pid_file(CF_PID_PATH)
    subprocess.run("pkill -f 'kohya_gui.py' || true", shell=True)
    subprocess.run("pkill -f 'cloudflared tunnel --url' || true", shell=True)


def is_port_free(port: int) -> bool:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        s.bind(("0.0.0.0", port))
        return True
    except OSError:
        return False
    finally:
        s.close()


def wait_port_open(proc, port: int, timeout: int) -> bool:
    for sec in range(1, max(1, timeout) + 1):
        if proc.poll() is not None:
            print(f"GUI exited early. returncode={proc.returncode}")
            return False

        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        try:
            ready = sock.connect_ex(("127.0.0.1", port)) == 0
        finally:
            sock.close()

        if ready:
            return True

        if sec % 5 == 0:
            print(f"waiting for GUI local port... {sec}s")
        time.sleep(1)

    return False


def ensure_cloudflared():
    if os.path.exists("/usr/local/bin/cloudflared"):
        return

    print("cloudflared 설치 중...")
    subprocess.run(
        "wget -q -O /tmp/cloudflared-linux-amd64.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        shell=True,
        check=True,
    )
    subprocess.run("dpkg -i /tmp/cloudflared-linux-amd64.deb", shell=True, check=True)
    print("cloudflared 설치 완료")


def launch_gui(config_path: str, port: int):
    if os.path.exists(GUI_LOG_PATH):
        os.remove(GUI_LOG_PATH)

    cmd = [
        "python", "-u", "kohya_gui.py",
        "--headless",
        "--listen", "0.0.0.0",
        "--server_port", str(port),
    ]

    if DISABLE_REQUIREMENTS_VERIFY:
        cmd.append("--noverify")

    if config_path and os.path.exists(config_path):
        cmd.extend(["--config", config_path])
        print("loading config:", config_path)
    else:
        print("starting GUI without preset config")

    logf = open(GUI_LOG_PATH, "w", encoding="utf-8", buffering=1)
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(cmd, cwd=KOHYA_DIR, env=env, stdout=logf, stderr=subprocess.STDOUT)

    with open(GUI_PID_PATH, "w", encoding="utf-8") as f:
        f.write(str(proc.pid))

    print(f"GUI started in background. pid={proc.pid} port={port}")
    print(f"gui log: {GUI_LOG_PATH}")
    return proc


def launch_cloudflared(port: int):
    if os.path.exists(CF_LOG_PATH):
        os.remove(CF_LOG_PATH)

    cmd = [
        "cloudflared", "tunnel",
        "--url", f"http://127.0.0.1:{port}",
        "--no-autoupdate",
        "--loglevel", "info",
    ]

    logf = open(CF_LOG_PATH, "w", encoding="utf-8", buffering=1)
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(cmd, env=env, stdout=logf, stderr=subprocess.STDOUT)

    with open(CF_PID_PATH, "w", encoding="utf-8") as f:
        f.write(str(proc.pid))

    print(f"cloudflared started. pid={proc.pid}")
    print(f"cloudflared log: {CF_LOG_PATH}")
    return proc


def find_cloudflare_url(log_text: str):
    urls = re.findall(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", log_text)
    return urls[-1] if urls else None


def wait_cloudflare_url(cf_proc, wait_seconds: int):
    found = None
    last_txt = ""

    for sec in range(1, max(1, wait_seconds) + 1):
        last_txt = read_text(CF_LOG_PATH)
        found = find_cloudflare_url(last_txt)
        if found:
            break

        if cf_proc.poll() is not None:
            print(f"cloudflared exited early. returncode={cf_proc.returncode}")
            break

        if sec % 5 == 0:
            print(f"waiting for Cloudflare URL... {sec}s")
        time.sleep(1)

    return found, last_txt


def run_once(config_path: str):
    for i in range(max(1, PORT_SCAN_TRIES)):
        port = int(GUI_PORT) + i
        if not is_port_free(port):
            print(f"port {port} is busy, try next")
            continue

        stop_prev_processes()
        gui_proc = launch_gui(config_path, port)

        if not wait_port_open(gui_proc, port, URL_WAIT_SECONDS):
            txt = read_text(GUI_LOG_PATH)
            if "Cannot find empty port in range" in txt or "Address already in use" in txt:
                print(f"port conflict detected at {port}, retry with next port")
                continue
            return None, port, txt, "gui"

        ensure_cloudflared()
        cf_proc = launch_cloudflared(port)
        url, cf_log = wait_cloudflare_url(cf_proc, URL_WAIT_SECONDS)

        if url:
            return url, port, cf_log, "ok"

        return None, port, cf_log, "cloudflare"

    return None, None, read_text(GUI_LOG_PATH), "gui"


initial_config = GUI_CONFIG_TO_LOAD.strip()
url, used_port, last_log, state = run_once(initial_config)

if (not url) and RETRY_WITHOUT_CONFIG_IF_FAIL and initial_config:
    print("Cloudflare URL not found. retry once without config...")
    url, used_port, last_log, state = run_once("")

if url:
    print(f"GUI URL (Cloudflare): {url}")
    print(f"used port: {used_port}")
else:
    print("Cloudflare URL not found yet.")
    if state == "gui":
        print("--- last 120 lines of /content/kohya_gui.log ---")
        txt = read_text(GUI_LOG_PATH)
    else:
        print("--- last 120 lines of /content/cloudflared.log ---")
        txt = read_text(CF_LOG_PATH)

    lines = txt.splitlines()
    print("\n".join(lines[-120:]))
    print("확인용: !tail -n 120 /content/kohya_gui.log")
    print("확인용: !tail -n 120 /content/cloudflared.log")


starting GUI without preset config
GUI started in background. pid=1800 port=7860
gui log: /content/kohya_gui.log
waiting for GUI local port... 5s
waiting for GUI local port... 10s
waiting for GUI local port... 15s
waiting for GUI local port... 20s
waiting for GUI local port... 25s
waiting for GUI local port... 30s
waiting for GUI local port... 35s
waiting for GUI local port... 40s
cloudflared 설치 중...
cloudflared 설치 완료
cloudflared started. pid=2036
cloudflared log: /content/cloudflared.log
waiting for Cloudflare URL... 5s
GUI URL (Cloudflare): https://string-aging-xbox-sail.trycloudflare.com
used port: 7860


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/kohya_ss_anima/logs --port 6007

In [ ]:
!tail -n 120 /content/kohya_gui.log

## 6. 학습/재개 경로 확인

학습 자체는 GUI에서 진행합니다.
아래 셀은 출력 경로/재개 경로 확인용입니다.


In [ ]:
#@title 6-1. 학습/재개 경로 확인 (JSON 자동 탐색 통합)
import glob
import os
import json

GUI_CONFIG_FOR_SUMMARY = ""  #@param {type:"string"}

def _load_json(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return json.load(f)
    except Exception:
        return None

def _pick_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

# 1) config json 선택
config_candidates = []
if GUI_CONFIG_FOR_SUMMARY.strip():
    config_candidates.append(GUI_CONFIG_FOR_SUMMARY.strip())
if "GUI_CONFIG_TO_LOAD" in globals() and str(GUI_CONFIG_TO_LOAD).strip():
    config_candidates.append(str(GUI_CONFIG_TO_LOAD).strip())
if "GUI_CONFIG_FOR_PATHS" in globals() and str(GUI_CONFIG_FOR_PATHS).strip():
    config_candidates.append(str(GUI_CONFIG_FOR_PATHS).strip())
config_candidates.extend(sorted(glob.glob("/content/last_*.json"), key=os.path.getmtime, reverse=True))
config_candidates.extend(sorted(glob.glob("/content/outputs/last_*.json"), key=os.path.getmtime, reverse=True))
config_candidates.extend(sorted(glob.glob("/content/kohya_ss_anima/outputs/last_*.json"), key=os.path.getmtime, reverse=True))

seen = set()
uniq = []
for p in config_candidates:
    if p not in seen:
        seen.add(p)
        uniq.append(p)
config_candidates = uniq

picked_config = _pick_existing(config_candidates)
cfg = _load_json(picked_config) if picked_config else {}

# 2) 경로 결정 (JSON 우선)
output_dir = cfg.get("output_dir", "") or "/content/kohya_ss_anima/outputs"
logging_dir = cfg.get("logging_dir", "") or "/content/kohya_ss_anima/logs"
train_data_dir = cfg.get("train_data_dir", "")
sample_prompts = cfg.get("sample_prompts", "")

# 3) 산출물 탐색
models = sorted(glob.glob(os.path.join(output_dir, "**/*.safetensors"), recursive=True), key=os.path.getmtime)
events = sorted(
    list(set(
        glob.glob(os.path.join(logging_dir, "**/events.out.tfevents*"), recursive=True)
        + glob.glob(os.path.join(output_dir, "**/events.out.tfevents*"), recursive=True)
        + glob.glob("/content/kohya_ss_anima/logs/**/events.out.tfevents*", recursive=True)
    )),
    key=os.path.getmtime,
)
states = sorted(
    glob.glob(os.path.join(output_dir, "**/*-state"), recursive=True)
    + glob.glob(os.path.join(output_dir, "**/*-state*"), recursive=True),
    key=os.path.getmtime,
)
run_tomls = sorted(glob.glob(os.path.join(output_dir, "**/config_lora-*.toml"), recursive=True), key=os.path.getmtime)
sample_imgs = sorted(glob.glob(os.path.join(output_dir, "sample/*.png")), key=os.path.getmtime)

print("[경로 요약]")
print("- picked config:", picked_config)
print("- output_dir:", output_dir, "exists=", os.path.exists(output_dir))
print("- logging_dir:", logging_dir, "exists=", os.path.exists(logging_dir))
print("- train_data_dir:", train_data_dir, "exists=", os.path.exists(train_data_dir) if train_data_dir else False)
print("- sample_prompts:", sample_prompts, "exists=", os.path.isfile(sample_prompts) if sample_prompts else False)

print("\n[산출물 요약]")
print(f"- models: {len(models)}")
print(f"- tfevents: {len(events)}")
print(f"- states: {len(states)}")
print(f"- run tomls: {len(run_tomls)}")
print(f"- sample images: {len(sample_imgs)}")
print("- latest model:", models[-1] if models else None)
print("- latest tfevents:", events[-1] if events else None)
print("- latest state:", states[-1] if states else None)
print("- latest run toml:", run_tomls[-1] if run_tomls else None)
print("- latest sample image:", sample_imgs[-1] if sample_imgs else None)


### 6-2. HF state 재개 시행착오 메모

HF state 이어학습 시 `--resume` 문자열 형식을 정확히 맞춰야 합니다.

**일반화된 표기**

```bash
--resume {HF_USERNAME}/{HF_REPO_NAME}/{PATH_IN_REPO_TO_STATE}:{REVISION}:{REPO_TYPE} --resume_from_huggingface
```

- `{PATH_IN_REPO_TO_STATE}` 예: `state/last-000003-state`
- `{REVISION}` 보통 `main`
- `{REPO_TYPE}` 보통 `model`

**실제 예시(현재 케이스)**

```bash
--resume Kamome33/vermouth_NAI/state/last-000003-state:main:model --resume_from_huggingface
```

**자주 나는 실수**

- 잘못된 예(경로 중복): `Kamome33/vermouth_NAI/vermouth_NAI/state/last-000003-state:main:model`
- `No files found in the specified repo id/path/revision` 오류가 나오면, 대부분 `repo_id/path_in_repo` 경계가 잘못된 경우입니다.


## 7. 결과 확인 및 HuggingFace Hub 업로드

학습 완료 후 결과물을 HuggingFace Hub에 업로드합니다.

- `7-1`: 모델 리포 업로드
- LoRA 체크포인트(`*.safetensors`), run TOML, GUI/JSON 설정, TensorBoard 로그를 업로드
- 경로는 `last_*.json`의 `output_dir`/`logging_dir`를 우선 사용(없으면 자동 탐색)
- `state` 폴더가 있으면 최신 state도 함께 업로드

- `7-2`: 데이터셋 리포 업로드(선택)
- `train_data_dir`를 `last_*.json`에서 자동으로 읽어 ZIP 업로드
- 필요하면 override 입력값으로 경로를 직접 지정 가능


In [ ]:
#@title 7-1. HuggingFace Hub 업로드 (JSON 경로 자동 반영)
# last_*.json에서 output/logging/train_data_dir 경로를 읽어 업로드 대상을 자동으로 구성합니다.
import glob
import os
import json
import shutil

GUI_CONFIG_FOR_PATHS = ""  #@param {type:"string"}
OUTPUT_DIR_OVERRIDE = ""  #@param {type:"string"}
LOG_DIR_OVERRIDE = ""  #@param {type:"string"}
UPLOAD_LATEST_STATE = True  #@param {type:"boolean"}
UPLOAD_PATH_SNAPSHOT = True  #@param {type:"boolean"}

def _load_json(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return json.load(f)
    except Exception:
        return None

def _pick_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def _artifact_score(base_dir):
    if not base_dir or not os.path.isdir(base_dir):
        return -1
    models = glob.glob(os.path.join(base_dir, "**/*.safetensors"), recursive=True)
    tomls = glob.glob(os.path.join(base_dir, "**/config_lora-*.toml"), recursive=True)
    return len(models) * 10 + len(tomls)

# 1) config json 자동 선택
config_candidates = []
if GUI_CONFIG_FOR_PATHS.strip():
    config_candidates.append(GUI_CONFIG_FOR_PATHS.strip())
if "GUI_CONFIG_TO_LOAD" in globals() and str(GUI_CONFIG_TO_LOAD).strip():
    config_candidates.append(str(GUI_CONFIG_TO_LOAD).strip())
config_candidates.extend(sorted(glob.glob("/content/last_*.json"), key=os.path.getmtime, reverse=True))
config_candidates.extend(sorted(glob.glob("/content/outputs/last_*.json"), key=os.path.getmtime, reverse=True))
config_candidates.extend(sorted(glob.glob("/content/kohya_ss_anima/outputs/last_*.json"), key=os.path.getmtime, reverse=True))

# 중복 제거
seen = set()
_uniq = []
for p in config_candidates:
    if p not in seen:
        seen.add(p)
        _uniq.append(p)
config_candidates = _uniq

picked_config = _pick_existing(config_candidates)
cfg = _load_json(picked_config) if picked_config else {}

cfg_output_dir = cfg.get("output_dir", "")
cfg_log_dir = cfg.get("logging_dir", "")
cfg_train_data_dir = cfg.get("train_data_dir", "")
cfg_reg_data_dir = cfg.get("reg_data_dir", "")
cfg_sample_prompts = cfg.get("sample_prompts", "")
cfg_dataset_config = cfg.get("dataset_config", "")

# 2) output/logging 경로 결정
output_candidates = []
if OUTPUT_DIR_OVERRIDE.strip():
    output_candidates.append(OUTPUT_DIR_OVERRIDE.strip())
if cfg_output_dir:
    output_candidates.append(cfg_output_dir)
output_candidates += [
    "/content/kohya_ss_anima/outputs",
    "/content/outputs",
    "/content/training_output",
]

log_candidates = []
if LOG_DIR_OVERRIDE.strip():
    log_candidates.append(LOG_DIR_OVERRIDE.strip())
if cfg_log_dir:
    log_candidates.append(cfg_log_dir)
log_candidates += [
    "/content/kohya_ss_anima/logs",
    "/content/logs",
    "/content/training_logs",
]

unique_output_candidates = []
for p in output_candidates:
    if p and p not in unique_output_candidates:
        unique_output_candidates.append(p)

# output_dir 우선순위: override > picked json > fallback 자동탐색
if OUTPUT_DIR_OVERRIDE.strip():
    OUTPUT_DIR = OUTPUT_DIR_OVERRIDE.strip()
elif cfg_output_dir and os.path.exists(cfg_output_dir):
    OUTPUT_DIR = cfg_output_dir
else:
    scored = sorted([(p, _artifact_score(p)) for p in unique_output_candidates], key=lambda x: x[1], reverse=True)
    OUTPUT_DIR = scored[0][0] if scored and scored[0][1] >= 0 else unique_output_candidates[0]

unique_log_candidates = []
for p in log_candidates:
    if p and p not in unique_log_candidates:
        unique_log_candidates.append(p)

# logging_dir 우선순위: override > picked json > fallback
if LOG_DIR_OVERRIDE.strip():
    LOG_DIR = LOG_DIR_OVERRIDE.strip()
elif cfg_log_dir and os.path.exists(cfg_log_dir):
    LOG_DIR = cfg_log_dir
else:
    LOG_DIR = _pick_existing(unique_log_candidates) or unique_log_candidates[0]

print("[경로 확인]")
print("- picked config:", picked_config)
print("- output_dir:", OUTPUT_DIR, "exists=", os.path.exists(OUTPUT_DIR))
print("- logging_dir:", LOG_DIR, "exists=", os.path.exists(LOG_DIR))
print("- train_data_dir:", cfg_train_data_dir, "exists=", os.path.exists(cfg_train_data_dir) if cfg_train_data_dir else False)
print("- reg_data_dir:", cfg_reg_data_dir, "exists=", os.path.exists(cfg_reg_data_dir) if cfg_reg_data_dir else False)
print("- sample_prompts:", cfg_sample_prompts, "exists=", os.path.isfile(cfg_sample_prompts) if cfg_sample_prompts else False)
print("- dataset_config:", cfg_dataset_config, "exists=", os.path.isfile(cfg_dataset_config) if cfg_dataset_config else False)

if not os.path.exists(OUTPUT_DIR):
    raise RuntimeError(f"Output dir not found: {OUTPUT_DIR}")

# 3) 산출물 수집
model_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**/*.safetensors"), recursive=True), key=os.path.getmtime)
run_tomls = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**/config_lora-*.toml"), recursive=True), key=os.path.getmtime)
json_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**/*.json"), recursive=True), key=os.path.getmtime)

if picked_config and os.path.isfile(picked_config):
    json_files.append(picked_config)
json_files = sorted(set(json_files), key=os.path.getmtime)

state_dirs = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "**/*-state"), recursive=True)
    + glob.glob(os.path.join(OUTPUT_DIR, "**/*-state*"), recursive=True),
    key=os.path.getmtime
)

gui_toml_patterns = [
    os.path.join(OUTPUT_DIR, "**/*.toml"),
    "/content/kohya_ss_anima/config.toml",
    "/content/kohya_ss_anima/*config*.toml",
    "/content/kohya_ss_anima/**/*config*.toml",
    "/content/kohya_ss_anima/dataset_config/**/*.toml",
]
gui_tomls = []
for p in gui_toml_patterns:
    gui_tomls.extend(glob.glob(p, recursive=True))
gui_tomls = sorted(set([p for p in gui_tomls if os.path.isfile(p)]), key=os.path.getmtime)

tb_patterns = [
    os.path.join(LOG_DIR, "**/events.out.tfevents*"),
    os.path.join(OUTPUT_DIR, "**/events.out.tfevents*"),
    "/content/kohya_ss_anima/logs/**/events.out.tfevents*",
    "/content/logs/**/events.out.tfevents*",
]
tb_files = []
for p in tb_patterns:
    tb_files.extend(glob.glob(p, recursive=True))
tb_files = sorted(set([p for p in tb_files if os.path.isfile(p)]), key=os.path.getmtime)

extra_cfg_files = []
for p in [cfg_sample_prompts, cfg_dataset_config]:
    if p and os.path.isfile(p):
        extra_cfg_files.append(p)

print(f"models: {len(model_files)}")
print(f"run tomls: {len(run_tomls)}")
print(f"gui tomls: {len(gui_tomls)}")
print(f"json files: {len(json_files)}")
print(f"tensorboard files: {len(tb_files)}")
print(f"state dirs: {len(state_dirs)}")
print(f"extra cfg files(from json): {len(extra_cfg_files)}")

if not model_files and not run_tomls:
    raise RuntimeError("No training artifacts found. Check output_dir path.")

if not HF_TOKEN:
    raise RuntimeError("HF token missing. Check cell 1-1.")
if not HF_USERNAME:
    raise RuntimeError("HF username missing. Check cell 1-1.")

from huggingface_hub import HfApi, CommitOperationAdd
api = HfApi()
repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
print(f"HF target: {repo_id}")
api.create_repo(repo_id=repo_id, repo_type="model", private=True, exist_ok=True)

operations = []
used_repo_paths = set()

def normalize_repo_path(path_in_repo: str) -> str:
    return path_in_repo.replace('\\', '/').lstrip('/')

def unique_repo_path(path_in_repo: str) -> str:
    rp = normalize_repo_path(path_in_repo)
    if rp not in used_repo_paths:
        used_repo_paths.add(rp)
        return rp
    base, ext = os.path.splitext(rp)
    idx = 2
    while True:
        cand = f"{base}__{idx}{ext}"
        if cand not in used_repo_paths:
            used_repo_paths.add(cand)
            return cand
        idx += 1

def add_file(local_path: str, path_in_repo: str):
    if not os.path.isfile(local_path):
        return
    rp = unique_repo_path(path_in_repo)
    operations.append(CommitOperationAdd(path_in_repo=rp, path_or_fileobj=local_path))

def rel_from(base: str, target: str) -> str:
    return os.path.relpath(target, base).replace('\\', '/')

for p in model_files:
    add_file(p, f"model/{rel_from(OUTPUT_DIR, p)}")
for p in run_tomls:
    add_file(p, f"config/run/{rel_from(OUTPUT_DIR, p)}")
for p in gui_tomls:
    if p.startswith('/content/'):
        add_file(p, f"config/gui/{p[len('/content/'):]}".replace('//', '/'))
    else:
        add_file(p, f"config/gui/{os.path.basename(p)}")
for p in json_files:
    if p.startswith('/content/'):
        add_file(p, f"config/json/{p[len('/content/'):]}".replace('//', '/'))
    else:
        add_file(p, f"config/json/{os.path.basename(p)}")
for p in tb_files:
    if p.startswith('/content/'):
        add_file(p, f"logs/{p[len('/content/'):]}".replace('//', '/'))
    else:
        add_file(p, f"logs/{os.path.basename(p)}")

for p in extra_cfg_files:
    add_file(p, f"config/from_json/{os.path.basename(p)}")

if os.path.exists('/content/pip_freeze.txt'):
    add_file('/content/pip_freeze.txt', 'config/pip_freeze.txt')

if UPLOAD_PATH_SNAPSHOT:
    snapshot = {
        "picked_config": picked_config,
        "resolved": {
            "output_dir": OUTPUT_DIR,
            "logging_dir": LOG_DIR,
            "train_data_dir": cfg_train_data_dir,
            "reg_data_dir": cfg_reg_data_dir,
            "sample_prompts": cfg_sample_prompts,
            "dataset_config": cfg_dataset_config,
        },
    }
    snap_path = '/content/upload_path_snapshot.json'
    with open(snap_path, 'w', encoding='utf-8') as f:
        json.dump(snapshot, f, ensure_ascii=False, indent=2)
    add_file(snap_path, 'config/path_snapshot.json')

if UPLOAD_LATEST_STATE and state_dirs:
    latest_state = state_dirs[-1]
    for root, _, files in os.walk(latest_state):
        for fn in files:
            lp = os.path.join(root, fn)
            add_file(lp, f"state/{os.path.basename(latest_state)}/{rel_from(latest_state, lp)}")
    print(f"including latest state: {latest_state}")

if not operations:
    raise RuntimeError("No files collected for upload.")

print(f"uploading {len(operations)} files in one commit...")
api.create_commit(
    repo_id=repo_id,
    repo_type="model",
    operations=operations,
    commit_message=f"Anima LoRA upload ({len(operations)} files)",
    token=HF_TOKEN,
)
print(f"upload complete: https://huggingface.co/{repo_id}")

# 선택: Google Drive 백업
if GDRIVE_OUTPUT_DIR:
    print(f"backup to Google Drive: {GDRIVE_OUTPUT_DIR}")
    if os.path.exists(GDRIVE_OUTPUT_DIR):
        shutil.rmtree(GDRIVE_OUTPUT_DIR)
    shutil.copytree(OUTPUT_DIR, GDRIVE_OUTPUT_DIR)
    backup_size_gb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fs in os.walk(GDRIVE_OUTPUT_DIR)
        for f in fs
    ) / 1024**3
    print(f"backup complete: {backup_size_gb:.2f} GB")


[경로 확인]
- picked config: /content/last_20260217-165318.json
- output_dir: /content/kohya_ss_anima/outputs exists= True
- logging_dir: /content/kohya_ss_anima/logs exists= True
- train_data_dir: /content/kohya_ss_anima/dataset/img exists= True
- reg_data_dir:  exists= False
- sample_prompts: shirakawa yuina, @WFS, blue headgear, platinum blonde hair, blue capelet, yuina eyelashes, 1girl, solo, outdoors, floating hair, sky, sunset, leaf, cloud, parted lips, open mouth, wind, portrait, falling leaves, blurry, dusk, orange sky, twilight, upper body, sunlight, cloudy sky, blurry foreground, lens flare --n low quality, worst quality, bad anatomy, --w 832 --h 1216 --d 1 --l 6.0 --s 28 exists= False
- dataset_config:  exists= False
models: 20
run tomls: 2
gui tomls: 3
json files: 2
tensorboard files: 1
state dirs: 0
extra cfg files(from json): 0
HF target: Kamome33/yuina_anima_minimal
uploading 30 files in one commit...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5626.ad944eadca12.14309.0:  84%|########3 | 60.7kB / 72.4kB            

  ...s/last-000001.safetensors:   9%|9         | 8.36MB / 91.9MB            

...

  ...s/last-000004.safetensors:   9%|9         | 8.36MB / 91.9MB            

  ...s/last-000005.safetensors:   9%|9         | 8.33MB / 91.9MB            

  ...s/last-000013.safetensors:   9%|9         | 8.34MB / 91.9MB            

  ...s/last-000014.safetensors:   9%|9         | 8.32MB / 91.9MB            

upload complete: https://huggingface.co/Kamome33/yuina_anima_minimal


In [ ]:
#@title 7-2. (선택) 데이터셋 HuggingFace 업로드 (JSON 경로 자동 반영)
# train_data_dir는 last_*.json에서 자동으로 읽고, 필요하면 override로 덮어쓸 수 있습니다.
import shutil
import os
import glob
import json

GUI_CONFIG_FOR_DATASET = ""  #@param {type:"string"}
DATASET_DIR_OVERRIDE = ""  #@param {type:"string"}
HF_DATASET_REPO_NAME = ""  #@param {type:"string"}
HF_DATASET_REPO_TYPE = "dataset"  #@param ["dataset", "model"]

def _load_json(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return json.load(f)
    except Exception:
        return None

def _pick_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

if not HF_TOKEN:
    print("HF 토큰이 설정되지 않았습니다. 셀 1-1을 확인하세요.")
    raise SystemExit

# 1) config json 선택
config_candidates = []
if GUI_CONFIG_FOR_DATASET.strip():
    config_candidates.append(GUI_CONFIG_FOR_DATASET.strip())
if "GUI_CONFIG_TO_LOAD" in globals() and str(GUI_CONFIG_TO_LOAD).strip():
    config_candidates.append(str(GUI_CONFIG_TO_LOAD).strip())
if "GUI_CONFIG_FOR_PATHS" in globals() and str(GUI_CONFIG_FOR_PATHS).strip():
    config_candidates.append(str(GUI_CONFIG_FOR_PATHS).strip())
config_candidates.extend(sorted(glob.glob("/content/last_*.json"), key=os.path.getmtime, reverse=True))
config_candidates.extend(sorted(glob.glob("/content/outputs/last_*.json"), key=os.path.getmtime, reverse=True))
config_candidates.extend(sorted(glob.glob("/content/kohya_ss_anima/outputs/last_*.json"), key=os.path.getmtime, reverse=True))

seen = set()
uniq = []
for p in config_candidates:
    if p not in seen:
        seen.add(p)
        uniq.append(p)
config_candidates = uniq

picked_config = _pick_existing(config_candidates)
cfg = _load_json(picked_config) if picked_config else {}

cfg_train_data_dir = cfg.get("train_data_dir", "")
cfg_dataset_source = globals().get("DATASET_SOURCE", "")

# 2) 데이터셋 경로 결정
LOCAL_DATASET_DIR = (DATASET_DIR_OVERRIDE.strip() if DATASET_DIR_OVERRIDE.strip() else cfg_train_data_dir) or "/content/dataset"

if not os.path.exists(LOCAL_DATASET_DIR):
    print(f"데이터셋 경로를 찾을 수 없습니다: {LOCAL_DATASET_DIR}")
    print(f"- picked config: {picked_config}")
    print(f"- cfg train_data_dir: {cfg_train_data_dir}")
    raise SystemExit

# 이미지 수(재귀)
img_exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".avif", ".jxl"}
img_count = 0
for root, _, files in os.walk(LOCAL_DATASET_DIR):
    for fn in files:
        if os.path.splitext(fn)[1].lower() in img_exts:
            img_count += 1

if not HF_DATASET_REPO_NAME.strip():
    HF_DATASET_REPO_NAME = f"{HF_REPO_NAME}_dataset"

from huggingface_hub import HfApi
api = HfApi()
dataset_repo_id = f"{HF_USERNAME}/{HF_DATASET_REPO_NAME}"

print("[경로 확인]")
print("- picked config:", picked_config)
print("- train_data_dir(from json):", cfg_train_data_dir)
print("- dataset dir(upload):", LOCAL_DATASET_DIR)
print("- image count:", img_count)

# 리포지토리 생성
api.create_repo(repo_id=dataset_repo_id, repo_type=HF_DATASET_REPO_TYPE, private=True, exist_ok=True)
print(f"데이터셋 리포: {dataset_repo_id} (type={HF_DATASET_REPO_TYPE})")

# ZIP 압축
print("\n데이터셋 ZIP 압축 중...")
zip_file = shutil.make_archive("/content/dataset_upload", 'zip', LOCAL_DATASET_DIR)
zip_size = os.path.getsize(zip_file) / 1024**2
print(f"- 압축 완료: {zip_size:.1f} MB")

# 업로드
print("업로드 중...")
api.upload_file(
    path_or_fileobj=zip_file,
    path_in_repo=f"{HF_DATASET_REPO_NAME}.zip",
    repo_id=dataset_repo_id,
    repo_type=HF_DATASET_REPO_TYPE,
    token=HF_TOKEN,
)

# README 업로드
readme_text = f"""# {HF_DATASET_REPO_NAME}\n\nLoRA 학습용 데이터셋\n\n- 원본 경로(DATASET_SOURCE): `{cfg_dataset_source}`\n- 사용 경로(train_data_dir): `{LOCAL_DATASET_DIR}`\n- 이미지 수(재귀): {img_count}\n- 선택된 설정 JSON: `{picked_config}`\n"""
readme_path = "/content/_dataset_readme.md"
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_text)
api.upload_file(
    path_or_fileobj=readme_path,
    path_in_repo="README.md",
    repo_id=dataset_repo_id,
    repo_type=HF_DATASET_REPO_TYPE,
    token=HF_TOKEN,
)

# 정리
try:
    os.remove(readme_path)
except Exception:
    pass
try:
    os.remove(zip_file)
except Exception:
    pass

print("\n" + "=" * 50)
print("데이터셋 업로드 완료!")
print(f"- https://huggingface.co/{dataset_repo_id}")
print(f"- {HF_DATASET_REPO_NAME}.zip ({zip_size:.1f} MB)")
print("=" * 50)


---

## 📌 트러블슈팅 & 팁

### VRAM OOM이 발생하면
1. `BLOCKS_TO_SWAP`을 `14`~`18`로 올리기 (최대 26)
2. `resolutions`를 `[512]`로 낮추기
3. `activation_checkpointing = 'unsloth'`로 변경 (더 공격적 절약)

### 학습이 너무 느리면
- `blocks_to_swap`을 줄이기 (VRAM이 여유 있다면)
- `gradient_accumulation_steps`를 2로 낮추기

### 과적합 체크
- Loss가 초반에 빠르게 떨어진 후 더 이상 감소하지 않으면 과적합 가능성
- `save_every_n_epochs = 5`로 줄여서 중간 체크포인트 저장
- ComfyUI에서 각 epoch의 LoRA를 직접 눈으로 비교하는 것이 가장 확실

### 캡션 관련
- Anima 태그 순서: `[quality/meta] [1girl/1boy] [character] [series] [artist] [general tags]`
- 아티스트 태그는 반드시 `@` 접두사: `@artistname`
- 캡션 부족 시 빈 캡션으로 학습됨 (경고만 출력)

### Colab 세션 끊김 대비
- `checkpoint_every_n_minutes = 30`이 설정되어 있음
- 끊기면 셀 1~4 재실행 후 **5-2 셀 (이어 학습)** 실행